In [ ]:
import os
import gradio as gr
from openai import OpenAI
import glob
import openai
import requests
from dotenv import load_dotenv 
from pathlib import Path

load_dotenv()

GEMINI_BASE_URL="https://generativelanguage.googleapis.com/v1beta/openai/"
google_api_key=os.getenv("GOOGLE_API_KEY")
gemini=OpenAI(api_key=google_api_key, base_url=GEMINI_BASE_URL)

sys = """
You represent Insurellm, the Insurance Tech company.
You are an expert in answering questions about Insurellm; its employees and its products.
You are provided with additional context that might be relevant to the user's question.
Give brief, accurate answers. If you don't know the answer, say so.

Relevant context:
"""


In [ ]:
know = {}

file = glob.glob("knowledge-base/employees/*")
for file in file:
    name = Path(file).stem.split(' ')[-1]
    with open(file, "r", encoding="utf-8") as f:
        know[name.lower()] = f.read()


In [ ]:
filenames = glob.glob("knowledge-base/products/*")

for filename in filenames:
    name = Path(filename).stem
    with open(filename, "r", encoding="utf-8") as f:
        know[name.lower()] = f.read()


In [ ]:
print(know)


In [ ]:
def relevent(mess):
    text = ''.join(ch for ch in mess if ch.isalpha() or ch.isspace())
    words = text.lower().split()
    return [know[word] for word in words if word in know]


In [ ]:
relevent("What is the role of Lancaster in the company?")


In [ ]:
def add(mess):
    relevant_info = relevent(mess)
    result = "The info is \n\n"
    result+= "\n\n".join(relevant_info)
    return result


In [ ]:
add("What is the role of Lancaster in the company?")


In [ ]:
add("What is carllm?")


In [ ]:
def chat(mess,history):
    system_message = sys + add(mess)
    message = [{"role": "system","content": system_message}] + history + [{"role": "user", "content": mess}]
    response = gemini.chat.completions.create(
        model = "gemini-2.5-flash",
        messages = message,
    )
    return response.choices[0].message.content


In [ ]:
gr.ChatInterface(fn=chat,
    title="RAG Chatbot",).launch(inbrowser=True)
